<a href="https://colab.research.google.com/github/jasminl/adaptics/blob/master/incremental.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Incremental generative learning

In [ ]:
import numpy as np

In [ ]:
def attempt_target(max_trial, bias, target, approx_threshold, inference_var):
  """ Main learning and inference function """
  for i in range(max_trial):

    # Generate new bias term
    new_bias = np.random.normal()

    loc = 0 + sum (bias) + new_bias

    # Inference
    y = np.random.normal(loc, inference_var)
    if np.abs(y - target) < approx_threshold:
      return True, new_bias, i
  return False, None, max_trial


In [1]:
# Key parameters
max_trial = 10
approx_threshold = 0.1  # How close we need to be to the target
inference_var = 0.1  # Variance of the genrative function
num_reps = 100

Run as a single process

In [8]:
target = 3
def single_process(target):
  iter_distr = []
  bias = []
  for i in range(num_reps):
    success, new_bias, iter = attempt_target(max_trial, bias, target,
                                             approx_threshold, inference_var)
    if success is True:
      iter_distr.append(iter)
  return iter_distr

successful_runs = single_process(target)
print('Number of successes: ', len(successful_runs))

Number of successes:  3


Run as multiple attempts with incremental targets

In [10]:

def multi_process(target):
  iter_distr = []
  bias_length = []

  split  = [1/3, 1/2, 3/4, 1]  # Last one always 1, the original goal

  for i in range(num_reps):

    bias = []  # Reset bias

    total_per_iter = 0
    for s in split:
      new_target = s * target
      success, new_bias, iter = attempt_target(max_trial, bias, new_target,
                                             approx_threshold, inference_var)
      total_per_iter += iter
      if new_bias is not None:
        bias.append(new_bias)

  #  print('Goal: ', new_target, ', bias: ', bias, 'iter: ', iter,
  #        'Success' if success else 'Failure')

  # Check if the original target is reached, if so, track num of iterations
    if success:
      iter_distr.append(total_per_iter)
      bias_length.append(len(bias))
  return iter_distr, bias_length

successful_runs, bias = multi_process(target)
print('Number successes: ', len(successful_runs))
#print(iter_distr)
#print(bias)

Number successes:  16
